# Pré-processamento

## Tratamento de Atributos

Tratamento dos atributos para que possam ser devidamente usadas pelos modelos posteriormente. 

In [ ]:
import pandas as pd
import glob
import os
ema_windows = [3, 5, 7, 10]
sma_windows = [3, 5, 7, 10]

import numpy as np

# ==========================================
# TARGET VARIABLE
# ==========================================

def get_target_class(df):    
    return df['FTR']

# ==========================================
# HISTORICAL POINTS FORM (Overall Momentum)
# ==========================================

def _build_points_timeline(df):
    """Calculates points earned per match to build a general form timeline."""
    
    # Home Team Points
    home = df[['Date', 'HomeTeam', 'FTR']].copy()
    home.columns = ['Date', 'Team', 'Result']
    home['Points'] = np.where(home['Result'] == 'H', 1, np.where(home['Result'] == 'D', 0, -1))
    
    # Away Team Points
    away = df[['Date', 'AwayTeam', 'FTR']].copy()
    away.columns = ['Date', 'Team', 'Result']
    away['Points'] = np.where(away['Result'] == 'A', 1, np.where(away['Result'] == 'D', 0, -1))
    
    # Drop the text result, keep just points
    home = home.drop('Result', axis=1)
    away = away.drop('Result', axis=1)
    
    return pd.concat([home, away]).sort_values('Date').reset_index(drop=True)

def get_home_team_points_avg(df, window=5):
    """Average points accumulated per game by the Home team over their last N matches."""
    timeline = _build_points_timeline(df)
    timeline['Points_Avg'] = timeline.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    merged = df.merge(timeline[['Date', 'Team', 'Points_Avg']], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left')
    return merged['Points_Avg']

def get_away_team_points_avg(df, window=5):
    """Average points accumulated per game by the Away team over their last N matches."""
    timeline = _build_points_timeline(df)
    timeline['Points_Avg'] = timeline.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    merged = df.merge(timeline[['Date', 'Team', 'Points_Avg']], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left')
    return merged['Points_Avg']

def get_specific_rolling_stats(df, team_col, stat_col, stat_prefix, sma_windows=[3, 5, 7, 10], ema_windows=[3, 5, 7, 10], min_matches=10):
    """
    Calculates specific form (Home playing at Home, or Away playing Away).
    The min_matches counter strictly counts games played in this specific context.
    If the stat_col represents points (1, 0, -1), it calculates the count of Wins, Draws, Losses and Streaks.
    """
    # 1. Isolate relevant columns
    temp_df = df[['Date', team_col, stat_col]].copy()
    
    # 2. Shift strictly within the specific team (Season isolation is handled upstream)
    temp_df['Shifted'] = temp_df.groupby(team_col)[stat_col].shift(1)
    
    # 3. This counter ticks up when they play at this specific venue
    temp_df['Prev_Count'] = temp_df.groupby(team_col).cumcount()

    result_df = pd.DataFrame(index=temp_df.index)

    # Check if we are calculating Form (which means we should also calculate W/D/L counts and streaks)
    is_form = stat_col in ['Home_Pts', 'Away_Pts']

    # STREAKS (Calculated efficiently on the shifted timeline using cumsum blocks)
    if is_form:
        # Win Streak
        is_win = temp_df['Shifted'] == 1
        win_blocks = (~is_win).groupby(temp_df[team_col]).cumsum()
        result_df[f'{team_col}_Spec_WinStreak_{stat_prefix}'] = is_win.groupby([temp_df[team_col], win_blocks]).cumcount()
        
        # Unbeaten Streak
        is_unbeaten = temp_df['Shifted'] >= 0
        unbeaten_blocks = (~is_unbeaten).groupby(temp_df[team_col]).cumsum()
        result_df[f'{team_col}_Spec_UnbeatenStreak_{stat_prefix}'] = is_unbeaten.groupby([temp_df[team_col], unbeaten_blocks]).cumcount()

        # Losing Streak
        is_loss = temp_df['Shifted'] == -1
        loss_blocks = (~is_loss).groupby(temp_df[team_col]).cumsum()
        result_df[f'{team_col}_Spec_LossStreak_{stat_prefix}'] = is_loss.groupby([temp_df[team_col], loss_blocks]).cumcount()

    # Calculate SMAs and EMAs for all requested windows
    for w in sma_windows:
        result_df[f'{team_col}_Spec_SMA_{w}_{stat_prefix}'] = temp_df.groupby(team_col)['Shifted'].transform(
            lambda x: x.rolling(window=w, min_periods=1).mean()
        )
        
        # If this is a form calculation, also get the strict counts of W/D/L in this window
        if is_form:
            result_df[f'{team_col}_Spec_Wins_{w}_{stat_prefix}'] = temp_df.groupby(team_col)['Shifted'].transform(
                lambda x: (x == 1).rolling(window=w, min_periods=1).sum()
            )
            result_df[f'{team_col}_Spec_Losses_{w}_{stat_prefix}'] = temp_df.groupby(team_col)['Shifted'].transform(
                lambda x: (x == -1).rolling(window=w, min_periods=1).sum()
            )
            result_df[f'{team_col}_Spec_Draws_{w}_{stat_prefix}'] = temp_df.groupby(team_col)['Shifted'].transform(
                lambda x: (x == 0).rolling(window=w, min_periods=1).sum()
            )

    for w in ema_windows:
        result_df[f'{team_col}_Spec_EMA_{w}_{stat_prefix}'] = temp_df.groupby(team_col)['Shifted'].transform(
            lambda x: x.ewm(span=w, min_periods=1).mean()
        )

    # Apply the strict purge rule
    cols_to_mask = [c for c in result_df.columns if 'SMA_' in c or 'EMA_' in c or 'Wins_' in c or 'Losses_' in c or 'Draws_' in c or 'Streak_' in c]
    result_df.loc[temp_df['Prev_Count'] < min_matches, cols_to_mask] = np.nan

    return result_df[cols_to_mask]


def get_advanced_rolling_stats(df, target_team, stat_prefix, home_stat_col, away_stat_col, sma_windows=[3, 5, 7, 10], ema_windows=[3, 5, 7, 10], min_matches=10):
    """
    Calculates General Form: SMAs and EMAs across all matches (Home + Away).
    If the stat represents points, calculates the count of Wins, Draws, Losses and Streaks.
    """
    # 1. Isolate Home matches
    home_df = df[['Date', 'HomeTeam', home_stat_col]].copy()
    home_df.columns = ['Date', 'Team', 'Stat']
    home_df['Original_Index'] = home_df.index
    home_df['Team_Type'] = 'Home'

    # 2. Isolate Away matches
    away_df = df[['Date', 'AwayTeam', away_stat_col]].copy()
    away_df.columns = ['Date', 'Team', 'Stat']
    away_df['Original_Index'] = away_df.index
    away_df['Team_Type'] = 'Away'

    # 3. Combine into a unified timeline
    timeline = pd.concat([home_df, away_df]).sort_values('Date')

    # 4. Shift the stat strictly within the Team group
    timeline['Shifted_Stat'] = timeline.groupby('Team')['Stat'].shift(1)
    
    # 5. Count how many previous matches have been played
    timeline['Previous_Matches_Count'] = timeline.groupby('Team').cumcount()

    # 6. Initialize the results dataframe
    result_df = pd.DataFrame(index=timeline.index)
    result_df['Original_Index'] = timeline['Original_Index']
    result_df['Team_Type'] = timeline['Team_Type']
    result_df['Previous_Matches_Count'] = timeline['Previous_Matches_Count']

    is_form = home_stat_col == 'Home_Pts' and away_stat_col == 'Away_Pts'

    # STREAKS (Calculated efficiently on the unified shifted timeline using cumsum blocks)
    if is_form:
        # Win Streak
        is_win = timeline['Shifted_Stat'] == 1
        win_blocks = (~is_win).groupby(timeline['Team']).cumsum()
        result_df[f'{target_team}_WinStreak_{stat_prefix}'] = is_win.groupby([timeline['Team'], win_blocks]).cumcount()
        
        # Unbeaten Streak
        is_unbeaten = timeline['Shifted_Stat'] >= 0
        unbeaten_blocks = (~is_unbeaten).groupby(timeline['Team']).cumsum()
        result_df[f'{target_team}_UnbeatenStreak_{stat_prefix}'] = is_unbeaten.groupby([timeline['Team'], unbeaten_blocks]).cumcount()

        # Losing Streak
        is_loss = timeline['Shifted_Stat'] == -1
        loss_blocks = (~is_loss).groupby(timeline['Team']).cumsum()
        result_df[f'{target_team}_LossStreak_{stat_prefix}'] = is_loss.groupby([timeline['Team'], loss_blocks]).cumcount()

    # 7. Calculate all SMAs and EMAs dynamically
    for w in sma_windows:
        # Simple Moving Average
        result_df[f'{target_team}_SMA_{w}_{stat_prefix}'] = timeline.groupby('Team')['Shifted_Stat'].transform(
            lambda x: x.rolling(window=w, min_periods=1).mean()
        )
        
        # Win/Draw/Loss Counts
        if is_form:
             result_df[f'{target_team}_Wins_{w}_{stat_prefix}'] = timeline.groupby('Team')['Shifted_Stat'].transform(
                 lambda x: (x == 1).rolling(window=w, min_periods=1).sum()
             )
             result_df[f'{target_team}_Losses_{w}_{stat_prefix}'] = timeline.groupby('Team')['Shifted_Stat'].transform(
                 lambda x: (x == -1).rolling(window=w, min_periods=1).sum()
             )
             result_df[f'{target_team}_Draws_{w}_{stat_prefix}'] = timeline.groupby('Team')['Shifted_Stat'].transform(
                 lambda x: (x == 0).rolling(window=w, min_periods=1).sum()
             )

    for w in ema_windows:
        # Exponential Moving Average
        result_df[f'{target_team}_EMA_{w}_{stat_prefix}'] = timeline.groupby('Team')['Shifted_Stat'].transform(
            lambda x: x.ewm(span=w, min_periods=1).mean()
        )

    # 8. Filter back to just the Home or Away rows we are calculating for
    result_df = result_df[result_df['Team_Type'] == target_team]

    # 9. ENFORCE THE 10 MATCH RULE
    cols_to_mask = [col for col in result_df.columns if 'SMA_' in col or 'EMA_' in col or 'Wins_' in col or 'Losses_' in col or 'Draws_' in col or 'Streak_' in col]
    result_df.loc[result_df['Previous_Matches_Count'] < min_matches, cols_to_mask] = np.nan

    # 10. Re-align perfectly with the original dataframe
    result_df = result_df.set_index('Original_Index').sort_index()

    return result_df[cols_to_mask]

# =========================================================
# 1. SINGLE SEASON PROCESSING ENGINE
# =========================================================
def process_single_season(df):
    """
    Takes a raw dataframe for a SINGLE season, calculates all rolling stats,
    formats dates, applies bookmaker odds, drops identifiers, and purges NaNs.
    """
    # Ensure Date is sorted chronologically within the season
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)
    df = df.sort_values(by='Date').reset_index(drop=True)

    df_season = pd.DataFrame()
    df_season['Date'] = df['Date']
    df_season['HomeTeam'] = df['HomeTeam']
    df_season['AwayTeam'] = df['AwayTeam']

    # --- CREATE PERSPECTIVE POINTS ---
    df['Home_Pts'] = df['FTR'].map({'H': 1, 'D': 0, 'A': -1})
    df['Away_Pts'] = df['FTR'].map({'H': -1, 'D': 0, 'A': 1})
    
    # 1. Goal Difference
    df['Home_GD'] = df['FTHG'] - df['FTAG']
    df['Away_GD'] = df['FTAG'] - df['FTHG']

    # 2. Clean Sheets & Failed To Score (FTS)
    df['Home_Clean_Sheet'] = (df['FTAG'] == 0).astype(int)
    df['Away_Clean_Sheet'] = (df['FTHG'] == 0).astype(int)
    df['Home_FTS'] = (df['FTHG'] == 0).astype(int)
    df['Away_FTS'] = (df['FTAG'] == 0).astype(int)

    # 3. Traditional Points (3 for Win, 1 for Draw, 0 for Loss)
    df['Home_Trad_Pts'] = df['FTR'].map({'H': 3, 'D': 1, 'A': 0})
    df['Away_Trad_Pts'] = df['FTR'].map({'H': 0, 'D': 1, 'A': 3})

    # 4. Mental Resilience (Comebacks from losing at Half-Time)
    # Check if HTR exists in the dataset first to prevent crashes on older datasets
    if 'HTR' in df.columns:
        df['Home_Comeback'] = ((df['HTR'] == 'A') & (df['FTR'].isin(['H', 'D']))).astype(int)
        df['Away_Comeback'] = ((df['HTR'] == 'H') & (df['FTR'].isin(['A', 'D']))).astype(int)

    new_feature_blocks = []

    # --- A. CALCULATE MATCH FORM (RESULTS) ---
    gen_home_form = get_advanced_rolling_stats(df, 'Home', 'Gen_Form', 'Home_Pts', 'Away_Pts', ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)
    gen_away_form = get_advanced_rolling_stats(df, 'Away', 'Gen_Form', 'Home_Pts', 'Away_Pts', ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)

    spec_home_form = get_specific_rolling_stats(df, 'HomeTeam', 'Home_Pts', 'Form', ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)
    spec_away_form = get_specific_rolling_stats(df, 'AwayTeam', 'Away_Pts', 'Form', ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)

    new_feature_blocks.extend([gen_home_form, gen_away_form, spec_home_form, spec_away_form])

    # --- B. CALCULATE STATISTICAL FORM ---
    stats_to_calculate = [
        ('Goals_Pro', 'FTHG', 'FTAG'),
        ('Goals_Suffered', 'FTAG', 'FTHG'), 
        ('Corners_Pro', 'HC', 'AC'),
        ('Corners_Against', 'AC', 'HC'),
        ('Fouls_Pro', 'HF', 'AF'),
        ('Fouls_Against', 'AF', 'HF'),       
        ('Yellows_Pro', 'HY', 'AY'),
        ('Yellows_Against', 'AY', 'HY'),     
        ('ShotsTarget_Pro', 'HST', 'AST'), 
        ('ShotsTarget_Against', 'AST', 'HST'),       
        ('Shots_Pro', 'HS', 'AS'), 
        ('Shots_Against', 'AS', 'HS'),       
        ('GD', 'Home_GD', 'Away_GD'),
        ('Clean_Sheet', 'Home_Clean_Sheet', 'Away_Clean_Sheet'),
        ('FTS', 'Home_FTS', 'Away_FTS'),
        ('Trad_Pts', 'Home_Trad_Pts', 'Away_Trad_Pts'),
        ('Comeback', 'Home_Comeback', 'Away_Comeback')
    ]

    for stat_prefix, h_col, a_col in stats_to_calculate:
        h_gen = get_advanced_rolling_stats(df, 'Home', f'Gen_{stat_prefix}', h_col, a_col, ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)
        a_gen = get_advanced_rolling_stats(df, 'Away', f'Gen_{stat_prefix}', h_col, a_col, ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)
        
        h_spec = get_specific_rolling_stats(df, 'HomeTeam', h_col, stat_prefix, ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)
        a_spec = get_specific_rolling_stats(df, 'AwayTeam', a_col, stat_prefix, ema_windows=ema_windows, sma_windows=sma_windows, min_matches=10)
        
        new_feature_blocks.extend([h_gen, a_gen, h_spec, a_spec])

    # Concatenate everything for this specific season
    df_season = pd.concat([df_season] + new_feature_blocks, axis=1)
    df_season['FTR'] = get_target_class(df)

    # --- C. TIME AND DATE LOGIC ---
    df_season['Week_of_Year'] = df['Date'].dt.isocalendar().week
    df_season['Day_of_Year'] = df['Date'].dt.dayofyear

    # --- D. BOOKMAKER ODDS ---
    home_bookies = ['1XBH', 'B365H', 'BFH', 'BFDH', 'BMGMH', 'BVH', 'BSH', 'BWH', 'CLH', 'GBH', 'IWH', 'LBH', 'PSH', 'PH', 'SOH', 'SBH', 'SJH', 'SYH', 'VCH', 'WHH']
    draw_bookies = ['1XBD', 'B365D', 'BFD', 'BFDD', 'BMGMD', 'BVD', 'BSD', 'BWD', 'CLD', 'GBD', 'IWD', 'LBD', 'PSD', 'PD', 'SOD', 'SBD', 'SJD', 'SYD', 'VCD', 'WHD']
    away_bookies = ['1XBA', 'B365A', 'BFA', 'BFDA', 'BMGMA', 'BVA', 'BSA', 'BWA', 'CLA', 'GBA', 'IWA', 'LBA', 'PSA', 'PA', 'SOA', 'SBA', 'SJA', 'SYA', 'VCA', 'WHA']

    valid_home = [c for c in home_bookies if c in df.columns]
    valid_draw = [c for c in draw_bookies if c in df.columns]
    valid_away = [c for c in away_bookies if c in df.columns]

    df_season['Avg_Odds_Home'] = df[valid_home].mean(axis=1)
    df_season['Avg_Odds_Draw'] = df[valid_draw].mean(axis=1)
    df_season['Avg_Odds_Away'] = df[valid_away].mean(axis=1)

    # --- E. FINAL GENERALIZATION & PURGE ---
    df_season = df_season.drop(columns=['Date', 'HomeTeam', 'AwayTeam'])
    
    # Drop NaNs purely for this season's 10-game warmup
    df_season = df_season.dropna().reset_index(drop=True)
    
    return df_season


# =========================================================
# 2. THE MAIN LOOP (FILE BY FILE)
# =========================================================

folder_path = "./raw-data/data/*.csv"     
all_files = glob.glob(folder_path)

processed_seasons_list = []

for filename in sorted(all_files):
    try:
        season_name = os.path.basename(filename).replace('.csv', '')
        print(f"Processing Season: {season_name}...")
        
        # Read the raw file
        temp_df = pd.read_csv(filename, encoding='unicode_escape')
        
        # Process the entire season in isolation
        season_processed_df = process_single_season(temp_df)
        
        # Add to our master list
        processed_seasons_list.append(season_processed_df)
        
    except pd.errors.ParserError as e:
        print(f"\n[!] PARSER ERROR IN FILE: {filename}")
        print(f"Error details: {e}\n")
        raise 

# =========================================================
# 3. MASTER CONCATENATION & EXPORT
# =========================================================
print("\nStitching all processed seasons together...")
df_train_final = pd.concat(processed_seasons_list, axis=0, ignore_index=True)

print(f"Final Dataset Ready! Total rows: {len(df_train_final)}")
df_train_final.to_csv('./processed-data/training_data.csv', index=False)
print("Advanced Training Data Exported Successfully!")

## Extração de Atributos


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def extraction():
    print("Loading dataset for Feature Extraction...")
    try:
        df = pd.read_csv('./processed-data/training_data.csv')    
    except FileNotFoundError:
        print("Error: Could not find './processed-data/training_data.csv'")
        return

    # Separate features and target
    X = df.drop(columns=['FTR'])
    y = df['FTR']
    
    print(f"Initial shape: {X.shape}")

    # Standardize the features (mean=0, variance=1) prior to PCA
    print("Applying StandardScaler...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fit PCA on the fully scaled dataset to determine variance ratios
    print("Fitting PCA to determine optimal components...")
    pca_full = PCA()
    pca_full.fit(X_scaled)

    # Calculate cumulative explained variance
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

    # Define the target variances requested by the professor
    target_variances = [0.90, 0.95, 0.99]

    for target_variance in target_variances:
        # Handle 100% variance carefully due to floating point precision limits
        if target_variance >= 1.00:
            optimal_components = len(cumulative_variance)
        else:
            optimal_components = np.argmax(cumulative_variance >= target_variance) + 1

        print(f"\n--- Target Variance: {target_variance*100:.0f}% ---")
        print(f"Original feature count: {X.shape[1]}")
        print(f"Components needed to retain {target_variance*100:.0f}% of variance: {optimal_components}")
        print(f"Dimensionality reduced by: {X.shape[1] - optimal_components} features")

        # Transform the dataset using the optimal number of components
        pca_optimal = PCA(n_components=optimal_components)
        X_pca = pca_optimal.fit_transform(X_scaled)

        # Reconstruct the DataFrame with principal components
        pca_columns = [f'PC_{i+1}' for i in range(optimal_components)]
        df_pca_final = pd.DataFrame(X_pca, columns=pca_columns, index=X.index)
        
        # Reattach the target variable
        df_pca_final['FTR'] = y.values

        # Save the transformed dataset with the variance percentage in the filename
        output_path = f'./processed-data/training_data_pca_{int(target_variance*100)}.csv'
        df_pca_final.to_csv(output_path, index=False)
        print(f"PCA Extracted Dataset saved to: {output_path}")

extraction()

## Seleção de Atributos 

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import RFECV, SelectKBest, mutual_info_classif
from sklearn.model_selection import StratifiedKFold
import time

def selection():
    print("Loading dataset for Feature Selection...")
    try:
        df = pd.read_csv('./processed-data/training_data.csv')
    except FileNotFoundError:
        print("Error: Could not find './processed-data/training_data.csv'")
        return

    # Separate features and target
    X = df.drop(columns=['FTR'])
    y = df['FTR']

    # Encode target variable ('A', 'D', 'H' -> 0, 1, 2)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    print(f"Initial feature count: {X.shape[1]}")

    # CRITICAL: SVC and KNN require scaled data to evaluate distances properly.
    # We scale the data for the selection process, but we will save the original unscaled values.
    print("Scaling data for geometric evaluation...")
    scaler = StandardScaler()
    X_scaled_array = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns, index=X.index)

    # Using StratifiedKFold to explicitly treat rows as independent
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Define the base estimators for each algorithm
    # SVC is now using 'rbf' exactly as it will in your final model pipeline
    models = {
        "XGBoost": XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42, n_jobs=-1),
        "RandomForest": RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1),        
    }

    print("\nStarting individual Feature Selection per algorithm...")

    for model_name, model in models.items():
        print(f"\n=======================================================")
        print(f" OPTIMIZING FEATURES FOR: {model_name}")
        print(f"=======================================================")
        start_time = time.time()        
        selector = RFECV(
            estimator=model,
            step=0.05,            # Drop bottom 5% of features each iteration
            cv=cv_strategy,
            scoring='accuracy',
            min_features_to_select=15,
            n_jobs=-1
        )
        selector.fit(X_scaled, y_encoded)
        selected_features = X.columns[selector.support_].tolist()
        optimal_num_features = selector.n_features_

        elapsed_time = time.time() - start_time
        print(f"-> Finished in {elapsed_time:.2f} seconds.")
        print(f"-> Optimal number of features found: {optimal_num_features}")
        
        # Save the optimized dataset using the ORIGINAL unscaled data
        X_optimal = X[selected_features]
        df_optimal = pd.concat([X_optimal, df['FTR']], axis=1)
        
        output_path = f'../processed-data/training_data_optimal_{model_name}.csv'
        df_optimal.to_csv(output_path, index=False)
        print(f"-> Dataset saved to: {output_path}")

    k_values = [25, 50, 100]

    print("\nStarting SelectKBest Feature Selection...")

    for k in k_values:
        print(f"\n=======================================================")
        print(f" OPTIMIZING FEATURES FOR: Top {k} Features")
        print(f"=======================================================")
        start_time = time.time()

        print(f"Method: Filter Method (SelectKBest with Mutual Information)")
        print(f"[!] Instantly calculating the top {k} features using Mutual Information statistics...")
        
        # Initialize and fit SelectKBest
        selector = SelectKBest(score_func=mutual_info_classif, k=k)
        selector.fit(X_scaled, y_encoded)
        
        selected_features = X.columns[selector.get_support()].tolist()
        optimal_num_features = len(selected_features)

        elapsed_time = time.time() - start_time
        print(f"-> Finished in {elapsed_time:.2f} seconds.")
        print(f"-> Optimal number of features found: {optimal_num_features}")
        
        # Save the optimized dataset using the ORIGINAL unscaled data
        X_optimal = X[selected_features]
        df_optimal = pd.concat([X_optimal, df['FTR']], axis=1)
        
        output_path = f'./processed-data/training_data_kbest_{k}.csv'
        df_optimal.to_csv(output_path, index=False)
        print(f"-> Dataset saved to: {output_path}")
selection()

# Predição

## Comparação geral
Fit de todos os modelos para cada uma das bases de dados, permitindo identificar o melhor. 

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, log_loss
import warnings
warnings.filterwarnings('ignore')

# Import all models
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

def compare_models():
    print("\n" + "="*70)
    print(" MASTER HYPERPARAMETER TUNING SUITE (MAX ACCURACY)")
    print("="*70)

    # 1. Define the models and their specific parameter grids
    models_config = {
        "XGBoost": {
            "estimator": XGBClassifier(objective='multi:softprob', num_class=3, random_state=42, n_jobs=-1),
            "needs_scaling": False,
            "params": {
                'model__n_estimators': [100, 200, 300, 500],
                'model__learning_rate': [0.01, 0.05, 0.1, 0.15],
                'model__max_depth': [3, 4, 5, 6],
                'model__subsample': [0.6, 0.8, 1.0],
                'model__colsample_bytree': [0.6, 0.8, 1.0],
                'model__gamma': [0, 0.1, 0.5, 1.0],
                'model__min_child_weight': [1, 3, 5]
            }
        },
        "RandomForest": {
            "estimator": RandomForestClassifier(random_state=42, n_jobs=-1),
            "needs_scaling": False,
            "params": {
                'model__n_estimators': [100, 200, 300, 500],
                'model__max_depth': [10, 20, 30, None],
                'model__min_samples_split': [2, 5, 10],
                'model__min_samples_leaf': [1, 2, 4],
                'model__max_features': ['sqrt', 'log2', None],
                'model__bootstrap': [True, False]
            }
        },
        "SVC": {
            "estimator": SVC(probability=True, random_state=42),
            "needs_scaling": True,
            "params": {
                'model__C': [0.1, 1, 10, 50],
                'model__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
                'model__kernel': ['rbf', 'poly'] # 'linear' is too slow for 50 iters
            }
        },
        "KNN": {
            "estimator": KNeighborsClassifier(n_jobs=-1),
            "needs_scaling": True,
            "params": {
                'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
                'model__weights': ['uniform', 'distance'],
                'model__metric': ['euclidean', 'manhattan', 'minkowski']
            }
        }
    }

    # 2. Define the datasets to test against
    base_path = '../processed-data/'
    datasets_to_check = [
        ("Raw", "training_data.csv"),
        ("KBest_25", "training_data_kbest_25.csv"),
        ("KBest_50", "training_data_kbest_50.csv"),
        ("KBest_100", "training_data_kbest_100.csv"),
        ("PCA_90", "training_data_pca_90.csv"),
        ("PCA_95", "training_data_pca_95.csv"),
        ("PCA_99", "training_data_pca_99.csv")
    ]

    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results_log = []

    for model_name, config in models_config.items():
        print(f"\n" + "="*50)
        print(f" COMMENCING TUNING FOR: {model_name}")
        print(f"="*50)

        # Add the model's specific optimal dataset to its checklist
        model_datasets = datasets_to_check.copy()
        model_datasets.append((f"Optimal_{model_name}", f"training_data_optimal_{model_name}.csv"))

        for data_label, filename in model_datasets:
            file_path = os.path.join(base_path, filename)
            
            if not os.path.exists(file_path):
                print(f"  [Skip] {data_label} not found ({filename})")
                continue

            print(f"\n  -> Evaluating on Dataset: {data_label}")
            
            df = pd.read_csv(file_path)
            X = df.drop(columns=['FTR'])
            y = df['FTR']

            le = LabelEncoder()
            y_encoded = le.fit_transform(y)
            
            # Split BEFORE tuning to evaluate real-world accuracy
            X_train, X_test, y_train, y_test = train_test_split(
                X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
            )

            # Build the pipeline depending on geometric vs tree model
            if config["needs_scaling"]:
                pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    ('model', config["estimator"])
                ])
            else:
                pipeline = Pipeline([
                    ('model', config["estimator"])
                ])

            # RandomizedSearchCV targeting Maximum Accuracy
            # Note: n_iter=20 is chosen because doing 4 models * 8 datasets = 32 tuning phases. 
            # 20 iterations keeps total runtime manageable.
            random_search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=config["params"],
                n_iter=20, 
                scoring='accuracy', 
                cv=cv_strategy,
                verbose=0,
                random_state=42,
                n_jobs=-1
            )

            print(f"     * Searching parameters...")
            random_search.fit(X_train, y_train)

            best_model = random_search.best_estimator_
            y_pred = best_model.predict(X_test)
            y_pred_probs = best_model.predict_proba(X_test)

            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='macro')
            lloss = log_loss(y_test, y_pred_probs)

            # Clean up the best params dictionary (remove 'model__' prefix)
            clean_params = {k.replace('model__', ''): v for k, v in random_search.best_params_.items()}

            print(f"     * Accuracy: {acc:.4f} | Macro F1: {f1:.4f} | Log Loss: {lloss:.4f}")
            
            # Save to log
            results_log.append({
                "Model": model_name,
                "Dataset": data_label,
                "Accuracy": acc,
                "Macro_F1": f1,
                "Log_Loss": lloss,
                "Best_Params": str(clean_params)
            })

    # Save all results to a CSV so you can easily review the best combinations
    if results_log:
        results_df = pd.DataFrame(results_log)
        # Sort by best accuracy overall
        results_df = results_df.sort_values(by="Accuracy", ascending=False)
        
        output_file = '../processed-data/master_tuning_results.csv'
        results_df.to_csv(output_file, index=False)
        
        print("\n" + "="*70)
        print(" ALL TUNING COMPLETE! ")
        print(f" Results mapped and saved to: {output_file}")
        print(" Top 5 Combinations found:")
        print(results_df[['Model', 'Dataset', 'Accuracy']].head(5).to_string(index=False))
        print("="*70)


compare_models()

## Random Forest
Random forest com seus hiperparâmetros e base de dados adequada, apenas análise mais aprofundada dos resultados obtidos. 

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

def evaluate_best_rf():
    print("\n" + "="*60)
    print(" EVALUATING: Optimized Random Forest on KBest_50")
    print("="*60)
    
    file_path = '../processed-data/training_data_kbest_50.csv'
    if not os.path.exists(file_path):
        print(f"Error: Could not find {file_path}")
        print("Make sure you have run the Feature Selection script first!")
        return

    # 1. Load and Prepare Data
    df = pd.read_csv(file_path)
    X = df.drop(columns=['FTR'])
    y = df['FTR']

    # 'A' becomes 0, 'D' becomes 1, 'H' becomes 2
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    # 2. Initialize Model with your Tuned Parameters
    # Random Forest DOES NOT need data scaling, so we feed it the raw data directly.
    print("Training Random Forest with optimal parameters...")
    model = RandomForestClassifier(
        n_estimators=500,
        min_samples_split=10,
        min_samples_leaf=2,
        max_features='sqrt',
        max_depth=10,
        bootstrap=True,
        random_state=42, 
        n_jobs=-1
    )

    # 3. Train and Predict
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_probs = model.predict_proba(X_test)

    # 4. Calculate Global Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    lloss = log_loss(y_test, y_pred_probs)
    
    # Calculate Precision and Recall per class (average=None returns an array for [A, D, H])
    precisions = precision_score(y_test, y_pred, average=None, zero_division=0)
    recalls = recall_score(y_test, y_pred, average=None, zero_division=0)
    
    print("\n--- GLOBAL TEST SET METRICS ---")
    print(f"Overall Accuracy: {acc * 100:.2f}%")    
    print(f"F1-Score:   {f1:.4f}")
    print(f"Log Loss:         {lloss:.4f}")
    
    print("\n--- PRECISION & RECALL ANALYSIS (TRUST METRICS) ---")
    print("Precision: When the model guesses a result, what is the % chance it is actually right?")
    print("Recall: Out of all the times this result actually happened, what % did the model catch?")
    # le.inverse_transform converts [0, 1, 2] back to ['A', 'D', 'H']
    for idx, class_name in enumerate(le.classes_):
        if class_name == 'A':
            label = "Away Win (A)"
        elif class_name == 'D':
            label = "Draw (D)"
        else:
            label = "Home Win (H)"
            
        print(f" -> {label}:")
        print(f"      Precision: {precisions[idx] * 100:.2f}%")
        print(f"      Recall:    {recalls[idx] * 100:.2f}%\n")

    # 5. Create the Confusion Matrix Graphic
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Greens', cbar=False, 
                annot_kws={"size": 14, "weight": "bold"})
    plt.title("Random Forest (KBest 50 Features)", fontsize=16, fontweight='bold', pad=15)
    plt.ylabel('True Result', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Result', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

evaluate_best_rf()